In [10]:
# CELL 1: Environment Setup and Library Installation
# This cell installs all required packages for Day 1 and sets up the environment

import os
import sys
import warnings
import subprocess
import importlib.util

# Suppress warnings for cleaner output
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")

# List of required packages with their import names and pip names
required_packages = [
    ("wikipediaapi", "wikipedia-api"),
    ("duckduckgo_search", "duckduckgo-search"),
    ("langchain", "langchain"),
    ("langchain_core", "langchain-core"),
    ("langchain_community", "langchain-community"),
    ("langgraph", "langgraph"),
    ("requests", "requests"),
    ("typing_extensions", "typing_extensions"),
    ("pydantic", "pydantic"),
    ("pickle", ""),  # Built-in, no installation needed
    ("json", ""),    # Built-in, no installation needed
]

def install_package(package_name):
    """Install a package using pip with quiet output."""
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package_name],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )
        return True
    except subprocess.CalledProcessError:
        return False

def check_and_install_packages():
    """Check if packages are installed, install missing ones."""
    installed = []
    failed = []
    
    for import_name, pip_name in required_packages:
        if not pip_name:  # Built-in packages
            continue
            
        # Check if package is installed
        spec = importlib.util.find_spec(import_name)
        if spec is None:
            print(f"Installing: {pip_name}")
            if install_package(pip_name):
                installed.append(pip_name)
            else:
                failed.append(pip_name)
        else:
            print(f"Already installed: {pip_name}")
    
    if installed:
        print(f"\nSuccessfully installed: {', '.join(installed)}")
    if failed:
        print(f"\nFailed to install: {', '.join(failed)}")
        print("Please install these manually using: pip install <package>")
    
    return len(failed) == 0

# Run package installation
print("Checking and installing required packages...")
print("-" * 50)
installation_success = check_and_install_packages()
print("-" * 50)

if installation_success:
    print("All packages are ready.")
else:
    print("Some packages failed to install. Please check the error messages above.")

# Verify critical package imports
print("\nVerifying critical imports...")
try:
    import wikipediaapi
    print("  wikipediaapi: OK")
except ImportError:
    print("  wikipediaapi: FAILED")
    
try:
    from duckduckgo_search import DDGS
    print("  duckduckgo-search: OK")
except ImportError:
    print("  duckduckgo-search: FAILED")
    
try:
    from langchain.tools import BaseTool
    print("  langchain: OK")
except ImportError:
    print("  langchain: FAILED")

print("\nEnvironment setup complete.")

Checking and installing required packages...
--------------------------------------------------
Already installed: wikipedia-api
Already installed: duckduckgo-search
Already installed: langchain
Already installed: langchain-core
Already installed: langchain-community
Already installed: langgraph
Already installed: requests
Already installed: typing_extensions
Already installed: pydantic
--------------------------------------------------
All packages are ready.

Verifying critical imports...
  wikipediaapi: OK
  duckduckgo-search: OK
  langchain: OK

Environment setup complete.


In [11]:
# CELL 2: Tool Definitions - Wikipedia API Tool (Fixed for API Compatibility)
# This cell defines the Wikipedia search and article retrieval tool with proper API usage

import wikipediaapi
from typing import Dict, Any, Optional, List
from langchain.tools import BaseTool
from pydantic import BaseModel, Field, PrivateAttr
import time
import re

class WikipediaInput(BaseModel):
    """Input schema for Wikipedia tool."""
    query: str = Field(description="The search query or article title to look up on Wikipedia")
    max_results: int = Field(default=3, description="Maximum number of search results to return")

class WikipediaTool(BaseTool):
    """Tool for searching and retrieving information from Wikipedia."""
    
    name: str = "wikipedia_search"
    description: str = "Search Wikipedia for information on any topic. Returns article summaries and details."
    args_schema: type[BaseModel] = WikipediaInput
    
    _user_agent: str = PrivateAttr(default="AgenticAI-Project/1.0 (Educational Purpose)")
    _wiki: Optional[wikipediaapi.Wikipedia] = PrivateAttr(default=None)
    _cache: Dict[str, Any] = PrivateAttr(default_factory=dict)
    
    def __init__(self):
        super().__init__()
        self._wiki = wikipediaapi.Wikipedia(
            language='en',
            user_agent=self._user_agent
        )
        self._cache = {}
    
    def _clean_text(self, text: str, max_length: int = 500) -> str:
        """Clean and truncate text."""
        if not text:
            return "No content available."
        
        text = re.sub(r'\n\s*\n', '\n\n', text)
        text = re.sub(r'\s+', ' ', text)
        
        if len(text) > max_length:
            text = text[:max_length] + "..."
        
        return text.strip()
    
    def search_articles(self, query: str, max_results: int = 3) -> List[Dict[str, Any]]:
        """Search for articles matching the query using Wikipedia API."""
        results = []
        
        try:
            # Use the Wikipedia API search correctly
            search_results = self._wiki.search(query)
            
            # Convert search results to list and limit
            search_list = list(search_results)
            limited_results = search_list[:max_results]
            
            for title in limited_results:
                page = self._wiki.page(title)
                if not page.exists():
                    continue
                    
                article_info = {
                    "title": page.title,
                    "summary": self._clean_text(page.summary, 300),
                    "url": page.fullurl,
                    "page_id": page.pageid
                }
                results.append(article_info)
                
                cache_key = f"article_{page.pageid}"
                self._cache[cache_key] = article_info
                
                time.sleep(0.1)
                
        except Exception as e:
            print(f"Wikipedia search error: {str(e)}")
            
        return results
    
    def get_article_by_title(self, title: str) -> Optional[Dict[str, Any]]:
        """Get a specific article by title."""
        cache_key = f"article_title_{title.lower()}"
        if cache_key in self._cache:
            return self._cache[cache_key]
        
        try:
            page = self._wiki.page(title)
            if not page.exists():
                return None
            
            # Get sections safely
            sections = []
            try:
                if hasattr(page, 'sections'):
                    sections = list(page.sections.keys()) if isinstance(page.sections, dict) else []
            except:
                sections = []
            
            article_info = {
                "title": page.title,
                "summary": self._clean_text(page.summary, 500),
                "content": self._clean_text(page.text, 1000),
                "url": page.fullurl,
                "page_id": page.pageid,
                "sections": sections
            }
            
            self._cache[cache_key] = article_info
            return article_info
            
        except Exception as e:
            print(f"Article retrieval error: {str(e)}")
            return None
    
    def _run(self, query: str, max_results: int = 3) -> str:
        """Execute the Wikipedia tool."""
        try:
            # First try to get as article title
            article = self.get_article_by_title(query)
            if article:
                return f"Wikipedia Article: {article['title']}\n\n{article['summary']}\n\nURL: {article['url']}"
            
            # Otherwise search
            search_results = self.search_articles(query, max_results)
            
            if not search_results:
                # Try a more general search with the query as a phrase
                try:
                    search_results = self.search_articles(query.lower(), max_results)
                except:
                    pass
                    
            if not search_results:
                return f"No Wikipedia results found for: {query}"
            
            output = f"Wikipedia Search Results for '{query}':\n\n"
            for idx, result in enumerate(search_results, 1):
                output += f"{idx}. {result['title']}\n"
                output += f"   {result['summary'][:200]}\n"
                output += f"   URL: {result['url']}\n\n"
            
            return output
            
        except Exception as e:
            return f"Error searching Wikipedia: {str(e)}"

def create_wikipedia_tool() -> WikipediaTool:
    """Factory function to create a Wikipedia tool instance."""
    return WikipediaTool()

# Test the tool
print("Testing Wikipedia Tool...")
tool = create_wikipedia_tool()

# Test search
result = tool._run("Artificial Intelligence")
print(result)
print("\n" + "="*50)
print("Wikipedia tool test completed.")

Testing Wikipedia Tool...
Wikipedia Article: Artificial intelligence

Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals. High-p...

URL: https://en.wikipedia.org/wiki/Artificial_intelligence

Wikipedia tool test completed.


In [20]:
# CELL 3: Tool Definitions - Web Search Tool (Using Wikipedia API with correct parameters)
# This cell defines the web search tool using Wikipedia API

import requests
import json
from typing import Dict, Any, Optional, List
from langchain.tools import BaseTool
from pydantic import BaseModel, Field, PrivateAttr
import re
import warnings
import urllib.parse

warnings.filterwarnings("ignore")

class WebSearchInput(BaseModel):
    """Input schema for web search tool."""
    query: str = Field(description="The search query to look up on the web")
    max_results: int = Field(default=3, description="Maximum number of search results to return")

class WebSearchTool(BaseTool):
    """Tool for searching the web using Wikipedia API."""
    
    name: str = "web_search"
    description: str = "Search for information on any topic. Returns titles and summaries."
    args_schema: type[BaseModel] = WebSearchInput
    
    _cache: Dict[str, Any] = PrivateAttr(default_factory=dict)
    
    def __init__(self):
        super().__init__()
        self._cache = {}
    
    def _clean_text(self, text: str, max_length: int = 300) -> str:
        """Clean and truncate text."""
        if not text:
            return "No content available."
        text = re.sub(r'<[^>]+>', '', text)
        text = re.sub(r'\s+', ' ', text)
        text = text.strip()
        if len(text) > max_length:
            text = text[:max_length] + "..."
        return text
    
    def search_wikipedia(self, query: str, max_results: int = 3) -> List[Dict[str, Any]]:
        """Search using Wikipedia API."""
        results = []
        try:
            # Wikipedia API endpoint
            url = "https://en.wikipedia.org/w/api.php"
            params = {
                "action": "query",
                "list": "search",
                "srsearch": query,
                "format": "json",
                "srlimit": max_results,
                "srprop": "snippet|titlesnippet"
            }
            
            headers = {
                "User-Agent": "AgenticAI-Project/1.0 (Educational Purpose)"
            }
            
            response = requests.get(url, params=params, headers=headers, timeout=10)
            
            if response.status_code == 200:
                data = response.json()
                search_results = data.get("query", {}).get("search", [])
                
                for item in search_results:
                    title = item.get("title", "")
                    snippet = item.get("snippet", "")
                    
                    # Get page URL
                    page_url = f"https://en.wikipedia.org/wiki/{urllib.parse.quote(title.replace(' ', '_'))}"
                    
                    results.append({
                        "title": self._clean_text(title, 100),
                        "snippet": self._clean_text(snippet, 300),
                        "url": page_url
                    })
                    
        except Exception as e:
            print(f"Wikipedia search error: {str(e)}")
        
        return results
    
    def search_web(self, query: str, max_results: int = 3) -> List[Dict[str, Any]]:
        """Perform web search."""
        results = []
        
        cache_key = f"search_{query.lower()}_{max_results}"
        if cache_key in self._cache:
            return self._cache[cache_key]
        
        results = self.search_wikipedia(query, max_results)
        
        if results:
            self._cache[cache_key] = results
        return results
    
    def _run(self, query: str, max_results: int = 3) -> str:
        """Execute the web search tool."""
        try:
            search_results = self.search_web(query, max_results)
            
            if not search_results:
                return f"No results found for: {query}"
            
            output = f"Search Results for '{query}':\n\n"
            for idx, result in enumerate(search_results, 1):
                output += f"{idx}. {result['title']}\n"
                output += f"   {result['snippet']}\n"
                if result['url']:
                    output += f"   URL: {result['url']}\n"
                output += "\n"
            
            return output
            
        except Exception as e:
            return f"Error performing search: {str(e)}"

def create_web_search_tool() -> WebSearchTool:
    """Factory function to create a web search tool instance."""
    return WebSearchTool()

# Test the tool with a direct API call to debug
print("Testing Wikipedia API directly...")
url = "https://en.wikipedia.org/w/api.php"
params = {
    "action": "query",
    "list": "search",
    "srsearch": "artificial intelligence",
    "format": "json",
    "srlimit": 3
}
response = requests.get(url, params=params, timeout=10)
if response.status_code == 200:
    data = response.json()
    print(f"API Response Status: Success")
    search_results = data.get("query", {}).get("search", [])
    print(f"Number of results: {len(search_results)}")
    for item in search_results:
        print(f"  - {item.get('title', 'No title')}")
else:
    print(f"API Error: {response.status_code}")

print("\n" + "="*50)

# Test the tool
print("\nTesting Web Search Tool...")
tool = create_web_search_tool()
result = tool._run("artificial intelligence", max_results=2)
print(result)
print("\n" + "="*50)
print("Web search tool test completed.")

Testing Wikipedia API directly...
API Error: 403


Testing Web Search Tool...
Search Results for 'artificial intelligence':

1. Artificial intelligence
   Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning
   URL: https://en.wikipedia.org/wiki/Artificial_intelligence

2. Artificial general intelligence
   Artificial general intelligence (AGI) is a hypothetical type of artificial intelligence that matches or surpasses human capabilities across virtually
   URL: https://en.wikipedia.org/wiki/Artificial_general_intelligence



Web search tool test completed.


In [21]:
# CELL 4: Tool Definitions - Calculator Tool
# This cell defines the calculator tool for performing mathematical operations

import re
import math
from typing import Dict, Any, Optional, List
from langchain.tools import BaseTool
from pydantic import BaseModel, Field, PrivateAttr

class CalculatorInput(BaseModel):
    """Input schema for calculator tool."""
    expression: str = Field(description="The mathematical expression to evaluate (e.g., '2 + 2', 'sqrt(16)', '10 * 5')")

class CalculatorTool(BaseTool):
    """Tool for performing mathematical calculations."""
    
    name: str = "calculator"
    description: str = "Perform mathematical calculations. Supports basic operations (+, -, *, /), exponentiation (**), and functions (sqrt, sin, cos, tan, log, log10)."
    args_schema: type[BaseModel] = CalculatorInput
    
    _cache: Dict[str, Any] = PrivateAttr(default_factory=dict)
    
    def __init__(self):
        super().__init__()
        self._cache = {}
        self._safe_functions = {
            'sqrt': math.sqrt,
            'sin': math.sin,
            'cos': math.cos,
            'tan': math.tan,
            'log': math.log,
            'log10': math.log10,
            'abs': abs,
            'ceil': math.ceil,
            'floor': math.floor,
            'round': round,
            'pi': math.pi,
            'e': math.e
        }
    
    def _validate_expression(self, expression: str) -> bool:
        """Validate that the expression contains only safe characters."""
        safe_pattern = r'^[\d\s+\-*/()\[\].,^%**a-zA-Z]+$'
        if not re.match(safe_pattern, expression):
            return False
        
        # Check for dangerous keywords
        dangerous_keywords = ['__', 'import', 'exec', 'eval', 'globals', 'locals', 'open', 'file']
        for keyword in dangerous_keywords:
            if keyword in expression.lower():
                return False
        
        return True
    
    def _evaluate_expression(self, expression: str) -> float:
        """Evaluate the mathematical expression safely."""
        # Replace ^ with ** for exponentiation
        expression = expression.replace('^', '**')
        
        # Create a safe evaluation context
        safe_context = {
            '__builtins__': {},
            'math': math,
            'sqrt': math.sqrt,
            'sin': math.sin,
            'cos': math.cos,
            'tan': math.tan,
            'log': math.log,
            'log10': math.log10,
            'abs': abs,
            'ceil': math.ceil,
            'floor': math.floor,
            'round': round,
            'pi': math.pi,
            'e': math.e
        }
        
        try:
            result = eval(expression, safe_context)
            return float(result)
        except Exception as e:
            raise ValueError(f"Error evaluating expression: {str(e)}")
    
    def _run(self, expression: str) -> str:
        """Execute the calculator tool."""
        try:
            # Clean the expression
            expression = expression.strip()
            
            if not expression:
                return "Error: No expression provided"
            
            # Validate expression
            if not self._validate_expression(expression):
                return f"Error: Invalid expression contains unsafe characters: {expression}"
            
            # Check cache
            cache_key = expression.lower()
            if cache_key in self._cache:
                return f"Result: {self._cache[cache_key]} (cached)"
            
            # Evaluate expression
            result = self._evaluate_expression(expression)
            
            # Format the result
            if result == int(result):
                formatted_result = str(int(result))
            else:
                formatted_result = f"{result:.6f}".rstrip('0').rstrip('.')
            
            # Cache the result
            self._cache[cache_key] = formatted_result
            
            return f"Result: {formatted_result}"
            
        except ValueError as e:
            return f"Error: {str(e)}"
        except ZeroDivisionError:
            return "Error: Division by zero"
        except Exception as e:
            return f"Error: {str(e)}"

def create_calculator_tool() -> CalculatorTool:
    """Factory function to create a calculator tool instance."""
    return CalculatorTool()

# Test the calculator tool
print("Testing Calculator Tool...")
tool = create_calculator_tool()

# Test various calculations
test_expressions = [
    "2 + 2",
    "10 * 5",
    "100 / 4",
    "2 ** 8",
    "sqrt(16)",
    "sin(pi/2)",
    "log(100)",
    "round(3.14159, 2)"
]

for expr in test_expressions:
    result = tool._run(expr)
    print(f"{expr} -> {result}")

print("\n" + "="*50)
print("Calculator tool test completed.")

Testing Calculator Tool...
2 + 2 -> Result: 4
10 * 5 -> Result: 50
100 / 4 -> Result: 25
2 ** 8 -> Result: 256
sqrt(16) -> Result: 4
sin(pi/2) -> Result: 1
log(100) -> Result: 4.60517
round(3.14159, 2) -> Result: 3.14

Calculator tool test completed.


In [25]:
# CELL 5: Tool Registry System (Fixed - No Pickle Issues)
# This cell creates the tool registry for managing and executing tools

from typing import Dict, Any, Optional, List
from langchain.tools import BaseTool
import time
from datetime import datetime
import json
import os

class ToolRegistry:
    """
    Registry for managing and executing tools.
    Provides functionality to register, retrieve, and execute tools.
    """
    
    def __init__(self):
        self._tools: Dict[str, BaseTool] = {}
        self._tool_metadata: Dict[str, Dict[str, Any]] = {}
        self._execution_history: List[Dict[str, Any]] = []
        
    def register_tool(self, tool: BaseTool, metadata: Optional[Dict[str, Any]] = None) -> None:
        """
        Register a tool in the registry.
        
        Args:
            tool: The tool instance to register
            metadata: Optional metadata about the tool
        """
        tool_name = tool.name
        
        if tool_name in self._tools:
            print(f"Warning: Tool '{tool_name}' already registered. Overwriting.")
        
        self._tools[tool_name] = tool
        
        self._tool_metadata[tool_name] = {
            "name": tool_name,
            "description": tool.description,
            "registered_at": datetime.now().isoformat(),
            "metadata": metadata or {}
        }
        
        print(f"Registered tool: {tool_name}")
    
    def register_tools(self, tools: List[BaseTool]) -> None:
        """
        Register multiple tools at once.
        
        Args:
            tools: List of tool instances to register
        """
        for tool in tools:
            self.register_tool(tool)
    
    def get_tool(self, tool_name: str) -> Optional[BaseTool]:
        """
        Get a tool by name.
        
        Args:
            tool_name: The name of the tool to retrieve
            
        Returns:
            The tool instance or None if not found
        """
        return self._tools.get(tool_name)
    
    def get_tool_info(self, tool_name: str) -> Optional[Dict[str, Any]]:
        """
        Get metadata for a tool.
        
        Args:
            tool_name: The name of the tool
            
        Returns:
            Tool metadata or None if not found
        """
        return self._tool_metadata.get(tool_name)
    
    def list_tools(self) -> List[Dict[str, Any]]:
        """
        List all registered tools with their information.
        
        Returns:
            List of tool information dictionaries
        """
        return [
            {
                "name": name,
                "description": tool.description,
                "metadata": self._tool_metadata.get(name, {})
            }
            for name, tool in self._tools.items()
        ]
    
    def execute_tool(self, tool_name: str, **kwargs) -> Dict[str, Any]:
        """
        Execute a registered tool with the given arguments.
        
        Args:
            tool_name: The name of the tool to execute
            **kwargs: Arguments to pass to the tool
            
        Returns:
            Dictionary containing execution results and metadata
        """
        start_time = time.time()
        
        result = {
            "tool_name": tool_name,
            "success": False,
            "result": None,
            "error": None,
            "execution_time": 0,
            "timestamp": datetime.now().isoformat()
        }
        
        tool = self.get_tool(tool_name)
        
        if not tool:
            result["error"] = f"Tool '{tool_name}' not found in registry"
            self._execution_history.append(result)
            return result
        
        try:
            output = tool._run(**kwargs)
            
            result["success"] = True
            result["result"] = output
            result["execution_time"] = time.time() - start_time
            
        except Exception as e:
            result["error"] = str(e)
            result["execution_time"] = time.time() - start_time
        
        self._execution_history.append(result)
        
        return result
    
    def execute_tool_with_retry(self, tool_name: str, max_retries: int = 3, **kwargs) -> Dict[str, Any]:
        """
        Execute a tool with retry logic.
        
        Args:
            tool_name: The name of the tool to execute
            max_retries: Maximum number of retry attempts
            **kwargs: Arguments to pass to the tool
            
        Returns:
            Dictionary containing execution results and metadata
        """
        attempts = 0
        last_result = None
        
        while attempts < max_retries:
            attempts += 1
            result = self.execute_tool(tool_name, **kwargs)
            
            if result["success"]:
                result["retry_count"] = attempts - 1
                return result
            
            last_result = result
            time.sleep(1)
        
        last_result["retry_count"] = attempts
        last_result["error"] = f"Failed after {attempts} attempts: {last_result.get('error', 'Unknown error')}"
        return last_result
    
    def get_execution_history(self, limit: int = 10) -> List[Dict[str, Any]]:
        """
        Get execution history.
        
        Args:
            limit: Maximum number of history entries to return
            
        Returns:
            List of execution history entries
        """
        return self._execution_history[-limit:]
    
    def clear_history(self) -> None:
        """Clear execution history."""
        self._execution_history = []
    
    def get_tool_usage_stats(self) -> Dict[str, Dict[str, Any]]:
        """
        Get usage statistics for all tools.
        
        Returns:
            Dictionary with tool usage statistics
        """
        stats = {}
        
        for tool_name in self._tools.keys():
            tool_executions = [
                entry for entry in self._execution_history 
                if entry["tool_name"] == tool_name
            ]
            
            successful = [e for e in tool_executions if e["success"]]
            
            stats[tool_name] = {
                "total_calls": len(tool_executions),
                "successful_calls": len(successful),
                "failed_calls": len(tool_executions) - len(successful),
                "success_rate": len(successful) / len(tool_executions) if tool_executions else 0,
                "avg_execution_time": sum(e["execution_time"] for e in tool_executions) / len(tool_executions) if tool_executions else 0
            }
        
        return stats
    
    def save_metadata(self, filepath: str = "day1_tool_metadata.json") -> None:
        """
        Save only metadata (not the tools themselves) to a JSON file.
        
        Args:
            filepath: Path to save the metadata
        """
        data = {
            "tool_metadata": self._tool_metadata,
            "execution_history": self._execution_history,
            "tool_names": list(self._tools.keys())
        }
        
        with open(filepath, 'w') as f:
            json.dump(data, f, indent=2)
        print(f"Tool metadata saved to {filepath}")
    
    def load_metadata(self, filepath: str = "day1_tool_metadata.json") -> None:
        """
        Load metadata from a JSON file.
        
        Args:
            filepath: Path to load the metadata from
        """
        if os.path.exists(filepath):
            with open(filepath, 'r') as f:
                data = json.load(f)
                self._tool_metadata = data.get("tool_metadata", {})
                self._execution_history = data.get("execution_history", [])
            print(f"Tool metadata loaded from {filepath}")
        else:
            print(f"File {filepath} not found")

def create_tool_registry() -> ToolRegistry:
    """Factory function to create a tool registry instance."""
    return ToolRegistry()

# Create registry
print("Creating Tool Registry...")
registry = create_tool_registry()

# Register tools using the classes defined in earlier cells
print("\nRegistering tools:")

# Get the classes from the current module
import sys
current_module = sys.modules[__name__]

# Check if tool classes are available and register them
registered_count = 0

if hasattr(current_module, 'WikipediaTool'):
    wikipedia_tool = WikipediaTool()
    registry.register_tool(wikipedia_tool)
    registered_count += 1

if hasattr(current_module, 'WebSearchTool'):
    web_tool = WebSearchTool()
    registry.register_tool(web_tool)
    registered_count += 1

if hasattr(current_module, 'CalculatorTool'):
    calc_tool = CalculatorTool()
    registry.register_tool(calc_tool)
    registered_count += 1

if registered_count == 0:
    print("  No tool classes found in namespace. Please run cells 2, 3, and 4 first.")
else:
    print(f"  Registered {registered_count} tools")

print("\nListing registered tools:")
tools = registry.list_tools()
if tools:
    for tool in tools:
        print(f"  - {tool['name']}: {tool['description'][:60]}...")
else:
    print("  No tools registered")

# Test execution if tools are available
if tools:
    print("\nTesting tool execution:")
    # Test calculator
    if registry.get_tool("calculator"):
        result = registry.execute_tool("calculator", expression="15 + 27")
        print(f"  calculator('15 + 27') -> {result['result']}")
    
    # Test Wikipedia
    if registry.get_tool("wikipedia_search"):
        result = registry.execute_tool("wikipedia_search", query="Python", max_results=1)
        if result['success']:
            print(f"  wikipedia_search('Python') -> {result['result'][:80]}...")

print("\n" + "="*50)
print("Tool registry setup completed.")

# Save metadata for Day 2
registry.save_metadata("day1_tool_metadata.json")
print("\nTool metadata saved successfully for Day 2.")

Creating Tool Registry...

Registering tools:
Registered tool: wikipedia_search
Registered tool: web_search
Registered tool: calculator
  Registered 3 tools

Listing registered tools:
  - wikipedia_search: Search Wikipedia for information on any topic. Returns artic...
  - web_search: Search for information on any topic. Returns titles and summ...
  - calculator: Perform mathematical calculations. Supports basic operations...

Testing tool execution:
  calculator('15 + 27') -> Result: 42
  wikipedia_search('Python') -> Wikipedia Article: Python

Python may refer to:

URL: https://en.wikipedia.org/w...

Tool registry setup completed.
Tool metadata saved to day1_tool_metadata.json

Tool metadata saved successfully for Day 2.


In [26]:
# CELL 6: Basic Agent Creation
# This cell creates a basic agent with tool calling capability

import json
import re
from typing import Dict, Any, Optional, List, Tuple
from langchain.tools import BaseTool
from datetime import datetime
import time

class BasicAgent:
    """
    A basic agent that can process queries and call tools.
    
    The agent receives a query, determines which tools to use,
    executes them, and returns a response.
    """
    
    def __init__(self, registry, name: str = "BasicAgent"):
        """
        Initialize the agent with a tool registry.
        
        Args:
            registry: ToolRegistry instance containing available tools
            name: Name of the agent
        """
        self.registry = registry
        self.name = name
        self.conversation_history = []
        self.max_iterations = 3
        
    def _parse_tool_call(self, text: str) -> Optional[Tuple[str, Dict[str, Any]]]:
        """
        Parse a tool call from text.
        
        Args:
            text: Text containing potential tool call
            
        Returns:
            Tuple of (tool_name, arguments) or None if no tool call found
        """
        # Pattern for tool calls: TOOL: tool_name(arg1=value1, arg2=value2)
        pattern = r'TOOL:\s*(\w+)\s*\((.*?)\)'
        match = re.search(pattern, text, re.IGNORECASE)
        
        if not match:
            return None
        
        tool_name = match.group(1)
        args_str = match.group(2)
        
        # Parse arguments
        args = {}
        if args_str.strip():
            # Simple key=value parsing
            arg_pattern = r'(\w+)\s*=\s*([^,]+)'
            for arg_match in re.finditer(arg_pattern, args_str):
                key = arg_match.group(1).strip()
                value = arg_match.group(2).strip()
                
                # Try to parse value as appropriate type
                if value.startswith('"') and value.endswith('"'):
                    value = value[1:-1]
                elif value.startswith("'") and value.endswith("'"):
                    value = value[1:-1]
                elif value.isdigit():
                    value = int(value)
                elif value.replace('.', '').isdigit():
                    value = float(value)
                elif value.lower() == 'true':
                    value = True
                elif value.lower() == 'false':
                    value = False
                elif value.lower() == 'none':
                    value = None
                
                args[key] = value
        
        return (tool_name, args)
    
    def _determine_tool_needs(self, query: str) -> List[str]:
        """
        Determine which tools might be needed for a query.
        
        Args:
            query: The user query
            
        Returns:
            List of tool names that might be useful
        """
        tool_needs = []
        query_lower = query.lower()
        
        # Check for calculator needs
        calc_keywords = ['calculate', 'compute', 'math', 'add', 'subtract', 'multiply', 'divide', 
                        'sqrt', 'sin', 'cos', 'tan', 'log', 'sum', 'average', 'pi', 'e']
        if any(keyword in query_lower for keyword in calc_keywords):
            tool_needs.append('calculator')
        
        # Check for Wikipedia needs
        wiki_keywords = ['wikipedia', 'wiki', 'encyclopedia', 'define', 'what is', 'who is', 'meaning of']
        if any(keyword in query_lower for keyword in wiki_keywords):
            tool_needs.append('wikipedia_search')
        
        # Check for web search needs (if query mentions current/recent/latest)
        web_keywords = ['latest', 'current', 'recent', 'news', 'today', 'update', 'search']
        if any(keyword in query_lower for keyword in web_keywords):
            tool_needs.append('web_search')
        
        # If no specific tool identified, default to web search for informational queries
        if not tool_needs and len(query.split()) > 2:
            tool_needs.append('web_search')
        
        return tool_needs
    
    def _execute_tools(self, tool_names: List[str], query: str) -> Dict[str, Any]:
        """
        Execute the specified tools with appropriate arguments.
        
        Args:
            tool_names: List of tool names to execute
            query: The original query
            
        Returns:
            Dictionary of tool results
        """
        results = {}
        
        for tool_name in tool_names:
            tool = self.registry.get_tool(tool_name)
            if not tool:
                results[tool_name] = {"error": f"Tool '{tool_name}' not found"}
                continue
            
            # Prepare arguments based on tool type
            kwargs = {}
            if tool_name == 'calculator':
                # Extract expression from query
                expr = re.sub(r'calculate|compute|what is|math', '', query).strip()
                kwargs['expression'] = expr if expr else query
            elif tool_name == 'wikipedia_search':
                kwargs['query'] = query
                kwargs['max_results'] = 2
            elif tool_name == 'web_search':
                kwargs['query'] = query
                kwargs['max_results'] = 3
            
            # Execute tool
            result = self.registry.execute_tool(tool_name, **kwargs)
            results[tool_name] = result
            
            # Add small delay between tool calls
            time.sleep(0.2)
        
        return results
    
    def _format_response(self, query: str, tool_results: Dict[str, Any]) -> str:
        """
        Format the response based on tool results.
        
        Args:
            query: The original query
            tool_results: Results from tool executions
            
        Returns:
            Formatted response string
        """
        # Check if any tool was successful
        successful_results = []
        for tool_name, result in tool_results.items():
            if result.get('success', False):
                successful_results.append((tool_name, result['result']))
        
        if not successful_results:
            return f"I couldn't find information about '{query}'. Please try rephrasing your query."
        
        # Build response
        response = f"Query: {query}\n\n"
        
        for tool_name, result in successful_results:
            response += f"[{tool_name}]\n"
            response += f"{result}\n"
            response += "-" * 40 + "\n"
        
        return response
    
    def process_query(self, query: str) -> str:
        """
        Process a user query and return a response.
        
        Args:
            query: The user query
            
        Returns:
            Formatted response from the agent
        """
        start_time = time.time()
        
        # Store query in history
        self.conversation_history.append({
            "timestamp": datetime.now().isoformat(),
            "role": "user",
            "content": query
        })
        
        # Determine which tools to use
        tool_names = self._determine_tool_needs(query)
        
        if not tool_names:
            response = f"I don't have any tools that can help with '{query}'. Please try a different query."
        else:
            # Execute tools
            tool_results = self._execute_tools(tool_names, query)
            
            # Format response
            response = self._format_response(query, tool_results)
        
        # Store response in history
        self.conversation_history.append({
            "timestamp": datetime.now().isoformat(),
            "role": "agent",
            "content": response,
            "execution_time": time.time() - start_time
        })
        
        return response
    
    def get_conversation_history(self) -> List[Dict[str, Any]]:
        """Get the conversation history."""
        return self.conversation_history
    
    def clear_history(self) -> None:
        """Clear the conversation history."""
        self.conversation_history = []
    
    def get_performance_stats(self) -> Dict[str, Any]:
        """
        Get performance statistics for the agent.
        
        Returns:
            Dictionary containing performance stats
        """
        agent_responses = [h for h in self.conversation_history if h['role'] == 'agent']
        
        if not agent_responses:
            return {"total_queries": 0}
        
        execution_times = [h.get('execution_time', 0) for h in agent_responses if 'execution_time' in h]
        
        return {
            "total_queries": len(agent_responses),
            "avg_execution_time": sum(execution_times) / len(execution_times) if execution_times else 0,
            "total_tools_used": len(set(
                h.get('tool_names', []) for h in self.conversation_history 
                if 'tool_names' in h
            ))
        }

def create_agent(registry, name: str = "BasicAgent") -> BasicAgent:
    """
    Factory function to create a basic agent.
    
    Args:
        registry: ToolRegistry instance
        name: Name of the agent
        
    Returns:
        BasicAgent instance
    """
    return BasicAgent(registry, name)

# Test the agent
print("Testing Basic Agent...")

# Get the registry from the previous cell
agent = create_agent(registry, "Day1Agent")

print("\nTest Query 1: Calculator")
response1 = agent.process_query("Calculate 25 * 4 + 10")
print(response1)

print("\nTest Query 2: Wikipedia")
response2 = agent.process_query("What is artificial intelligence")
print(response2[:300] + "...")

print("\nTest Query 3: Mixed")
response3 = agent.process_query("What is machine learning and calculate 100/4")
print(response3[:300] + "...")

print("\nPerformance Stats:")
stats = agent.get_performance_stats()
print(f"  Total queries: {stats['total_queries']}")
print(f"  Average execution time: {stats['avg_execution_time']:.2f} seconds")

print("\n" + "="*50)
print("Basic agent test completed.")

# Save agent data for Day 2
agent_data = {
    "agent_name": agent.name,
    "conversation_history": agent.conversation_history,
    "tool_names": list(registry._tools.keys()),
    "performance_stats": agent.get_performance_stats()
}

import json
with open("day1_agent_data.json", "w") as f:
    json.dump(agent_data, f, indent=2)

print("\nAgent data saved to day1_agent_data.json for Day 2.")

Testing Basic Agent...

Test Query 1: Calculator
Query: Calculate 25 * 4 + 10

[calculator]
Error: Error evaluating expression: invalid syntax (<string>, line 1)
----------------------------------------


Test Query 2: Wikipedia
Wikipedia search error: 'SearchResults' object is not iterable
Wikipedia search error: 'SearchResults' object is not iterable
Query: What is artificial intelligence

[calculator]
Error: Error evaluating expression: invalid syntax (<string>, line 1)
----------------------------------------
[wikipedia_search]
No Wikipedia results found for: What is artificial intelligence
----------------------------------------
...

Test Query 3: Mixed
Wikipedia search error: 'SearchResults' object is not iterable
Wikipedia search error: 'SearchResults' object is not iterable
Query: What is machine learning and calculate 100/4

[calculator]
Error: Error evaluating expression: invalid syntax (<string>, line 1)
----------------------------------------
[wikipedia_search]
No Wikiped

In [27]:
# CELL 6: Basic Agent Creation (Fixed)
# This cell creates a basic agent with tool calling capability

import json
import re
from typing import Dict, Any, Optional, List, Tuple
from langchain.tools import BaseTool
from datetime import datetime
import time

class BasicAgent:
    """
    A basic agent that can process queries and call tools.
    
    The agent receives a query, determines which tools to use,
    executes them, and returns a response.
    """
    
    def __init__(self, registry, name: str = "BasicAgent"):
        """
        Initialize the agent with a tool registry.
        
        Args:
            registry: ToolRegistry instance containing available tools
            name: Name of the agent
        """
        self.registry = registry
        self.name = name
        self.conversation_history = []
        self.max_iterations = 3
        
    def _determine_tool_needs(self, query: str) -> List[Tuple[str, Dict[str, Any]]]:
        """
        Determine which tools might be needed for a query and prepare arguments.
        
        Args:
            query: The user query
            
        Returns:
            List of tuples (tool_name, arguments)
        """
        tool_calls = []
        query_lower = query.lower()
        
        # Check for calculator needs - extract mathematical expression
        calc_patterns = [
            r'calculate\s+([\d+\-*/()\s.]+)',
            r'what is\s+([\d+\-*/()\s.]+)',
            r'compute\s+([\d+\-*/()\s.]+)',
            r'([\d+\-*/()\s.]+)'
        ]
        
        for pattern in calc_patterns:
            match = re.search(pattern, query_lower)
            if match:
                expr = match.group(1).strip()
                # Check if it looks like a math expression
                if re.search(r'[\d+\-*/]', expr) and not re.search(r'[a-zA-Z]', expr.replace('sqrt', '').replace('sin', '').replace('cos', '').replace('tan', '').replace('log', '')):
                    tool_calls.append(('calculator', {'expression': expr}))
                    break
        
        # Check for Wikipedia needs - extract topic
        wiki_patterns = [
            r'what is\s+([a-zA-Z\s]+)',
            r'who is\s+([a-zA-Z\s]+)',
            r'define\s+([a-zA-Z\s]+)',
            r'wikipedia\s+([a-zA-Z\s]+)',
            r'about\s+([a-zA-Z\s]+)'
        ]
        
        # Always add Wikipedia search for informational queries
        if not tool_calls:
            for pattern in wiki_patterns:
                match = re.search(pattern, query_lower)
                if match:
                    topic = match.group(1).strip()
                    if topic and len(topic) > 2:
                        tool_calls.append(('wikipedia_search', {'query': topic, 'max_results': 2}))
                        break
        
        # If no tool identified yet, try to extract the main topic
        if not tool_calls:
            words = query.split()
            if len(words) > 1:
                # Remove common stopwords and use remaining as topic
                stopwords = ['what', 'is', 'are', 'the', 'of', 'and', 'to', 'for', 'in', 'on', 'at', 'with', 'by']
                topic_words = [w for w in words if w.lower() not in stopwords]
                if topic_words:
                    topic = ' '.join(topic_words[:3])  # Use first 3 significant words
                    if len(topic) > 3:
                        tool_calls.append(('wikipedia_search', {'query': topic, 'max_results': 2}))
        
        return tool_calls
    
    def _execute_tools(self, tool_calls: List[Tuple[str, Dict[str, Any]]]) -> Dict[str, Any]:
        """
        Execute the specified tools with their arguments.
        
        Args:
            tool_calls: List of tuples (tool_name, arguments)
            
        Returns:
            Dictionary of tool results
        """
        results = {}
        
        for tool_name, kwargs in tool_calls:
            tool = self.registry.get_tool(tool_name)
            if not tool:
                results[tool_name] = {"error": f"Tool '{tool_name}' not found", "success": False}
                continue
            
            result = self.registry.execute_tool(tool_name, **kwargs)
            results[tool_name] = result
            
            time.sleep(0.2)
        
        return results
    
    def _format_response(self, query: str, tool_results: Dict[str, Any]) -> str:
        """
        Format the response based on tool results.
        
        Args:
            query: The original query
            tool_results: Results from tool executions
            
        Returns:
            Formatted response string
        """
        # Check if any tool was successful
        successful_results = []
        for tool_name, result in tool_results.items():
            if result.get('success', False) and result.get('result'):
                successful_results.append((tool_name, result['result']))
        
        if not successful_results:
            error_messages = []
            for tool_name, result in tool_results.items():
                if result.get('error'):
                    error_messages.append(f"{tool_name}: {result['error']}")
            if error_messages:
                return f"Could not process query '{query}'. Errors: {', '.join(error_messages)}"
            return f"I couldn't find information about '{query}'. Please try rephrasing your query."
        
        # Build response
        response = f"Query: {query}\n\n"
        
        for tool_name, result in successful_results:
            response += f"[{tool_name}]\n"
            response += f"{result}\n"
            response += "-" * 40 + "\n"
        
        return response
    
    def process_query(self, query: str) -> str:
        """
        Process a user query and return a response.
        
        Args:
            query: The user query
            
        Returns:
            Formatted response from the agent
        """
        start_time = time.time()
        
        # Store query in history
        self.conversation_history.append({
            "timestamp": datetime.now().isoformat(),
            "role": "user",
            "content": query
        })
        
        # Determine which tools to use with arguments
        tool_calls = self._determine_tool_needs(query)
        
        if not tool_calls:
            response = f"I don't have any tools that can help with '{query}'. Please try a different query."
        else:
            # Execute tools
            tool_results = self._execute_tools(tool_calls)
            
            # Format response
            response = self._format_response(query, tool_results)
        
        # Store response in history
        self.conversation_history.append({
            "timestamp": datetime.now().isoformat(),
            "role": "agent",
            "content": response,
            "execution_time": time.time() - start_time
        })
        
        return response
    
    def get_conversation_history(self) -> List[Dict[str, Any]]:
        """Get the conversation history."""
        return self.conversation_history
    
    def clear_history(self) -> None:
        """Clear the conversation history."""
        self.conversation_history = []
    
    def get_performance_stats(self) -> Dict[str, Any]:
        """Get performance statistics for the agent."""
        agent_responses = [h for h in self.conversation_history if h['role'] == 'agent']
        
        if not agent_responses:
            return {"total_queries": 0}
        
        execution_times = [h.get('execution_time', 0) for h in agent_responses if 'execution_time' in h]
        
        return {
            "total_queries": len(agent_responses),
            "avg_execution_time": sum(execution_times) / len(execution_times) if execution_times else 0
        }

def create_agent(registry, name: str = "BasicAgent") -> BasicAgent:
    """Factory function to create a basic agent."""
    return BasicAgent(registry, name)

# Test the agent
print("Testing Basic Agent...")

agent = create_agent(registry, "Day1Agent")

print("\nTest Query 1: Calculator")
response1 = agent.process_query("Calculate 25 * 4 + 10")
print(response1)

print("\nTest Query 2: Wikipedia")
response2 = agent.process_query("What is artificial intelligence")
print(response2)

print("\nTest Query 3: Calculator and Wikipedia mixed")
response3 = agent.process_query("What is machine learning and calculate 100 divided by 4")
print(response3)

print("\nPerformance Stats:")
stats = agent.get_performance_stats()
print(f"  Total queries: {stats['total_queries']}")
print(f"  Average execution time: {stats['avg_execution_time']:.2f} seconds")

print("\n" + "="*50)
print("Basic agent test completed.")

# Save agent data for Day 2
agent_data = {
    "agent_name": agent.name,
    "conversation_history": agent.conversation_history,
    "tool_names": list(registry._tools.keys()),
    "performance_stats": agent.get_performance_stats()
}

with open("day1_agent_data.json", "w") as f:
    json.dump(agent_data, f, indent=2)

print("\nAgent data saved to day1_agent_data.json for Day 2.")

Testing Basic Agent...

Test Query 1: Calculator
Query: Calculate 25 * 4 + 10

[calculator]
Result: 110
----------------------------------------


Test Query 2: Wikipedia
Query: What is artificial intelligence

[wikipedia_search]
Wikipedia Article: Artificial intelligence

Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals. High-p...

URL: https://en.wikipedia.org/wiki/Artificial_intelligence
----------------------------------------


Test Query 3: Calculator and Wikipedia mixed
Query: What is machine learning and calculate 100 divided by 4

[calculato

In [28]:
# CELL 7: Testing and Evaluation
# This cell tests the agent with various queries and measures performance

import json
import time
from datetime import datetime
from typing import Dict, Any, List

class AgentTester:
    """
    Test harness for evaluating agent performance.
    """
    
    def __init__(self, agent):
        """
        Initialize the tester with an agent.
        
        Args:
            agent: The BasicAgent instance to test
        """
        self.agent = agent
        self.test_results = []
        self.test_queries = [
            # Calculator queries
            {"query": "Calculate 15 + 30", "expected_tool": "calculator"},
            {"query": "What is 200 / 5", "expected_tool": "calculator"},
            {"query": "Compute 12 * 12", "expected_tool": "calculator"},
            
            # Wikipedia queries
            {"query": "What is Python programming", "expected_tool": "wikipedia_search"},
            {"query": "Who is Albert Einstein", "expected_tool": "wikipedia_search"},
            {"query": "Define machine learning", "expected_tool": "wikipedia_search"},
            
            # Mixed queries
            {"query": "What is deep learning and calculate 50 + 30", "expected_tool": "mixed"},
            {"query": "About quantum computing and 10 * 10", "expected_tool": "mixed"},
        ]
    
    def run_tests(self) -> List[Dict[str, Any]]:
        """
        Run all tests and collect results.
        
        Returns:
            List of test results
        """
        print("Running test suite...")
        print("=" * 60)
        
        for idx, test in enumerate(self.test_queries, 1):
            print(f"\nTest {idx}: {test['query']}")
            print("-" * 40)
            
            start_time = time.time()
            response = self.agent.process_query(test['query'])
            execution_time = time.time() - start_time
            
            # Determine which tool was actually used
            tools_used = []
            for tool_name in self.agent.registry._tools.keys():
                if tool_name in response:
                    tools_used.append(tool_name)
            
            success = len(tools_used) > 0
            
            result = {
                "test_id": idx,
                "query": test['query'],
                "expected_tool": test['expected_tool'],
                "tools_used": tools_used,
                "success": success,
                "execution_time": execution_time,
                "response_length": len(response),
                "timestamp": datetime.now().isoformat()
            }
            
            self.test_results.append(result)
            
            # Print summary
            print(f"Tools used: {tools_used if tools_used else 'None'}")
            print(f"Success: {success}")
            print(f"Execution time: {execution_time:.3f}s")
            print(f"Response preview: {response[:150]}...")
        
        return self.test_results
    
    def get_summary_stats(self) -> Dict[str, Any]:
        """
        Get summary statistics from test results.
        
        Returns:
            Dictionary of summary statistics
        """
        if not self.test_results:
            return {"error": "No test results available"}
        
        total_tests = len(self.test_results)
        successful_tests = sum(1 for r in self.test_results if r['success'])
        avg_time = sum(r['execution_time'] for r in self.test_results) / total_tests
        
        # Tool usage statistics
        tool_usage = {}
        for result in self.test_results:
            for tool in result['tools_used']:
                tool_usage[tool] = tool_usage.get(tool, 0) + 1
        
        return {
            "total_tests": total_tests,
            "successful_tests": successful_tests,
            "success_rate": successful_tests / total_tests * 100,
            "average_execution_time": avg_time,
            "tool_usage": tool_usage,
            "fastest_test": min(r['execution_time'] for r in self.test_results),
            "slowest_test": max(r['execution_time'] for r in self.test_results)
        }
    
    def generate_report(self) -> str:
        """
        Generate a formatted test report.
        
        Returns:
            Formatted report string
        """
        stats = self.get_summary_stats()
        
        report = []
        report.append("=" * 60)
        report.append("AGENT TESTING REPORT")
        report.append("=" * 60)
        report.append(f"Generated: {datetime.now().isoformat()}")
        report.append("")
        
        report.append("OVERALL STATISTICS:")
        report.append("-" * 40)
        report.append(f"Total tests: {stats['total_tests']}")
        report.append(f"Successful tests: {stats['successful_tests']}")
        report.append(f"Success rate: {stats['success_rate']:.1f}%")
        report.append(f"Average execution time: {stats['average_execution_time']:.3f}s")
        report.append(f"Fastest test: {stats['fastest_test']:.3f}s")
        report.append(f"Slowest test: {stats['slowest_test']:.3f}s")
        report.append("")
        
        report.append("TOOL USAGE STATISTICS:")
        report.append("-" * 40)
        for tool, count in stats['tool_usage'].items():
            report.append(f"  {tool}: {count} times")
        report.append("")
        
        report.append("DETAILED TEST RESULTS:")
        report.append("-" * 40)
        for result in self.test_results:
            status = "PASS" if result['success'] else "FAIL"
            report.append(f"Test {result['test_id']}: {status}")
            report.append(f"  Query: {result['query']}")
            report.append(f"  Tools used: {', '.join(result['tools_used']) if result['tools_used'] else 'None'}")
            report.append(f"  Time: {result['execution_time']:.3f}s")
            report.append("")
        
        report.append("=" * 60)
        report.append("END OF REPORT")
        report.append("=" * 60)
        
        return "\n".join(report)

def create_tester(agent) -> AgentTester:
    """Factory function to create an agent tester."""
    return AgentTester(agent)

# Run tests
print("Creating test harness...")
tester = create_tester(agent)

# Execute tests
test_results = tester.run_tests()

print("\n" + "=" * 60)
print("\nGenerating summary report...")
print(tester.generate_report())

# Save test results
test_data = {
    "test_results": test_results,
    "summary": tester.get_summary_stats(),
    "agent_name": agent.name,
    "timestamp": datetime.now().isoformat()
}

with open("day1_test_results.json", "w") as f:
    json.dump(test_data, f, indent=2)

print("\nTest results saved to day1_test_results.json")

# Final verification - check success criteria
print("\n" + "=" * 60)
print("DAY 1 SUCCESS CRITERIA VERIFICATION")
print("=" * 60)

stats = tester.get_summary_stats()

criteria = [
    ("Wikipedia search returns real results", "wikipedia_search" in stats['tool_usage']),
    ("Calculator handles basic arithmetic", "calculator" in stats['tool_usage']),
    ("Agent processes queries with tool calls", stats['successful_tests'] > 0),
    ("Performance < 2s per query", stats['average_execution_time'] < 2.0)
]

for criterion, status in criteria:
    print(f"{'[PASS]' if status else '[FAIL]'} {criterion}")

print("\n" + "=" * 60)
print("Day 1 testing complete.")

Creating test harness...
Running test suite...

Test 1: Calculate 15 + 30
----------------------------------------
Tools used: ['calculator']
Success: True
Execution time: 0.200s
Response preview: Query: Calculate 15 + 30

[calculator]
Result: 45
----------------------------------------
...

Test 2: What is 200 / 5
----------------------------------------
Tools used: ['calculator']
Success: True
Execution time: 0.200s
Response preview: Query: What is 200 / 5

[calculator]
Result: 40
----------------------------------------
...

Test 3: Compute 12 * 12
----------------------------------------
Tools used: ['calculator']
Success: True
Execution time: 0.200s
Response preview: Query: Compute 12 * 12

[calculator]
Result: 144
----------------------------------------
...

Test 4: What is Python programming
----------------------------------------
Tools used: ['wikipedia_search']
Success: True
Execution time: 0.641s
Response preview: Query: What is Python programming

[wikipedia_search]
Wikipe

In [30]:
# CELL 8: Save Day 1 Data for Day 2
# This cell saves all necessary data for Day 2

import json
import pickle
import os
from datetime import datetime

print("Saving Day 1 Data for Day 2...")
print("=" * 60)

# Create data directory if it doesn't exist
if not os.path.exists("day1_data"):
    os.makedirs("day1_data")

# 1. Save the registry metadata (JSON)
registry_data = {
    "tool_names": list(registry._tools.keys()),
    "tool_metadata": registry._tool_metadata,
    "execution_history": registry._execution_history
}

with open("day1_data/registry_metadata.json", "w") as f:
    json.dump(registry_data, f, indent=2)
print("  Saved registry_metadata.json")

# 2. Save agent data (JSON)
agent_data = {
    "agent_name": agent.name,
    "conversation_history": agent.conversation_history,
    "performance_stats": agent.get_performance_stats()
}

with open("day1_data/agent_data.json", "w") as f:
    json.dump(agent_data, f, indent=2)
print("  Saved agent_data.json")

# 3. Save test results (already saved, but copy to day1_data)
import shutil
if os.path.exists("day1_test_results.json"):
    shutil.copy("day1_test_results.json", "day1_data/test_results.json")
    print("  Saved test_results.json")

# 4. Create a summary file for Day 2
summary_data = {
    "day": 1,
    "completed_at": datetime.now().isoformat(),
    "tools_available": list(registry._tools.keys()),
    "agent_name": agent.name,
    "test_results": {
        "total_tests": len(tester.test_results),
        "successful_tests": sum(1 for r in tester.test_results if r['success']),
        "average_execution_time": tester.get_summary_stats()['average_execution_time']
    },
    "files_created": [
        "day1_data/registry_metadata.json",
        "day1_data/agent_data.json",
        "day1_data/test_results.json",
        "day1_data/summary.json"
    ]
}

with open("day1_data/summary.json", "w") as f:
    json.dump(summary_data, f, indent=2)
print("  Saved summary.json")

# 5. Create tool instances for Day 2 (can't pickle tools, so we recreate them)
tool_classes_data = {
    "wikipedia_tool": {
        "class_name": "WikipediaTool",
        "module": "__main__"
    },
    "web_search_tool": {
        "class_name": "WebSearchTool",
        "module": "__main__"
    },
    "calculator_tool": {
        "class_name": "CalculatorTool",
        "module": "__main__"
    }
}

with open("day1_data/tool_classes.json", "w") as f:
    json.dump(tool_classes_data, f, indent=2)
print("  Saved tool_classes.json")

print("\n" + "=" * 60)
print("DAY 1 DATA SUMMARY")
print("=" * 60)

print("\nTools available for Day 2:")
for tool_name in registry._tools.keys():
    print(f"  - {tool_name}")

print("\nConversation history:")
for entry in agent.conversation_history[-4:]:  # Show last 4 entries
    role = entry['role']
    content = entry['content'][:80] + "..." if len(entry['content']) > 80 else entry['content']
    print(f"  {role}: {content}")

print("\nPerformance stats:")
stats = agent.get_performance_stats()
print(f"  Total queries processed: {stats['total_queries']}")
print(f"  Average execution time: {stats['avg_execution_time']:.3f}s")

print("\n" + "=" * 60)
print("Day 1 data successfully saved. Ready for Day 2.")
print("\nTo start Day 2, run: day2_planning_memory.ipynb")
print("Files saved in: day1_data/")

# Verify all files were created
print("\nVerifying saved files:")
files_to_verify = [
    "day1_data/registry_metadata.json",
    "day1_data/agent_data.json",
    "day1_data/test_results.json",
    "day1_data/summary.json",
    "day1_data/tool_classes.json"
]

for filepath in files_to_verify:
    if os.path.exists(filepath):
        size = os.path.getsize(filepath)
        print(f"  [OK] {filepath} ({size} bytes)")
    else:
        print(f"  [MISSING] {filepath}")

print("\n" + "=" * 60)
print("Day 1 Complete!")

Saving Day 1 Data for Day 2...
  Saved registry_metadata.json
  Saved agent_data.json
  Saved test_results.json
  Saved summary.json
  Saved tool_classes.json

DAY 1 DATA SUMMARY

Tools available for Day 2:
  - wikipedia_search
  - web_search
  - calculator

Conversation history:
  user: What is deep learning and calculate 50 + 30
  agent: Query: What is deep learning and calculate 50 + 30

[calculator]
Result: 80
----...
  user: About quantum computing and 10 * 10
  agent: Query: About quantum computing and 10 * 10

[wikipedia_search]
No Wikipedia resu...

Performance stats:
  Total queries processed: 11
  Average execution time: 0.444s

Day 1 data successfully saved. Ready for Day 2.

To start Day 2, run: day2_planning_memory.ipynb
Files saved in: day1_data/

Verifying saved files:
  [OK] day1_data/registry_metadata.json (7629 bytes)
  [OK] day1_data/agent_data.json (6903 bytes)
  [OK] day1_data/test_results.json (2921 bytes)
  [OK] day1_data/summary.json (464 bytes)
  [OK] day1_data